# 03 — Evaluation, Robustness & Quantization
**Plant Disease Detection (Residual CBM)** — Step 3 of 4

Self-contained notebook: no `scripts/` import. This is the resume-facing
evaluation notebook — beyond plain accuracy it reports:

- Per-class precision/recall/F1 + confusion matrix
- **ROC-AUC and PR-AUC** (one-vs-rest, macro-averaged)
- **Top-2 / Top-3 accuracy**
- **Concept-head evaluation** (the 4-unit auxiliary head is otherwise never scored)
- **Cohen's Kappa & Matthews Correlation Coefficient** — needed to precisely
  characterize how (not) informative the corrupted-image predictions are
- **95% bootstrap confidence interval** on test accuracy
- **Calibration** (reliability diagram + Brier score) — is the model's
  confidence trustworthy, not just its argmax?
- **INT8 vs FP32 latency benchmark** — the actual edge-deployment payoff

**Prerequisites:** `best_model.keras` present (run `02_Model_Training.ipynb` first).


In [ ]:
import os
import time
import json
import ast

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve, top_k_accuracy_score,
    cohen_kappa_score, matthews_corrcoef, hamming_loss, f1_score,
    precision_score, recall_score, brier_score_loss,
)
from sklearn.preprocessing import label_binarize

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
sns.set_style("whitegrid")

IMG_SIZE = 224
SEED = 42
SELECTED_CLASSES = [
    "Pepper__bell___Bacterial_spot", "Pepper__bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Tomato_Bacterial_spot",
    "Tomato_Early_blight", "Tomato_Late_blight", "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot", "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot", "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus", "Tomato_healthy",
]
CONCEPT_NAMES = ["Is Healthy", "Has Spots", "Is Blighted", "Complex Pathogen"]
N_CLASSES = len(SELECTED_CLASSES)

manifest_csv_path = "manifest.csv" if os.path.exists("manifest.csv") else "../manifest.csv"
model_path = "best_model.keras" if os.path.exists("best_model.keras") else "../best_model.keras"

os.makedirs("eval_report" if os.path.exists("manifest.csv") else "../eval_report", exist_ok=True)

manifest = pd.read_csv(manifest_csv_path)
manifest["concept_vec"] = manifest["concept_vec"].apply(ast.literal_eval)
test_df = manifest[manifest["split"] == "test"]
print(f"[INFO] Test split: {len(test_df)} rows across {test_df['Class_Name'].nunique()} classes")

model = tf.keras.models.load_model(model_path)
print(f"[INFO] Loaded {model_path} ({model.count_params():,} params)")


## 3.0 Self-contained tf.data pipeline + full-inference helper
Runs the model once over a dataset and returns every raw array the metrics
below need (`y_true`, `y_pred`, `y_prob`, concept true/pred) — recomputed
fresh rather than reloaded from a cached JSON, since the extended metrics
below need full probability arrays that a plain accuracy report doesn't carry.

In [ ]:
def _decode_resize(path, img_size=IMG_SIZE):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [img_size, img_size])
    return img

def get_tf_dataset(df, batch_size=32, augment_fn=None):
    paths = df["File_Path"].values
    concepts = np.stack(df["concept_vec"].values).astype(np.float32)
    labels = df["label_idx"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, concepts, labels))

    def _load(path, concept, label):
        img = _decode_resize(path)
        if augment_fn is not None:
            img = augment_fn(img)
        return img, (concept, label)

    return ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

def run_inference(model, ds):
    """Full forward pass -> raw arrays for every metric in this notebook."""
    y_true, y_pred, y_prob = [], [], []
    concept_true, concept_prob = [], []
    for imgs, (concepts, labels) in ds:
        c_pred, d_pred = model.predict(imgs, verbose=0)
        y_true.append(labels.numpy())
        y_prob.append(d_pred)
        y_pred.append(np.argmax(d_pred, axis=1))
        concept_true.append(concepts.numpy())
        concept_prob.append(c_pred)
    return {
        "y_true": np.concatenate(y_true),
        "y_pred": np.concatenate(y_pred),
        "y_prob": np.concatenate(y_prob),
        "concept_true": np.concatenate(concept_true),
        "concept_prob": np.concatenate(concept_prob),
    }

print("[OPERATION] Running full inference over the clean test set...")
ds_clean = get_tf_dataset(test_df)
clean = run_inference(model, ds_clean)
print(f"[DONE] {len(clean['y_true'])} predictions collected.")


## 3.1 Clean test accuracy + per-class report

In [ ]:
clean_accuracy = float(np.mean(clean["y_pred"] == clean["y_true"]))
print(f"[RESULT] Clean test accuracy: {clean_accuracy:.4f}  (n={len(clean['y_true'])})")

report_dict = classification_report(clean["y_true"], clean["y_pred"], target_names=SELECTED_CLASSES,
                                     output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
print("\nPer-class precision / recall / F1:")
print(report_df.round(3))


## 3.2 Confusion matrix

In [ ]:
cm = confusion_matrix(clean["y_true"], clean["y_pred"])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=False, cmap="Blues", xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES,
            cbar_kws={"label": "row-normalized proportion"}, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized) — clean test set")
plt.xticks(rotation=75, ha="right"); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

most_confused = []
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i != j and cm[i, j] > 0:
            most_confused.append((SELECTED_CLASSES[i], SELECTED_CLASSES[j], int(cm[i, j])))
most_confused = sorted(most_confused, key=lambda x: -x[2])[:5]
print("[RESULT] Top confused (true -> predicted) pairs:")
for true_c, pred_c, n in most_confused:
    print(f"  {true_c}  ->  {pred_c}   ({n} images)")


## 3.3 ROC-AUC & PR-AUC (one-vs-rest, macro-averaged)
Accuracy alone hides how well-separated the predicted probabilities are.
Macro-averaged ROC-AUC and PR-AUC (average precision) treat every class
equally regardless of support — useful given the class-imbalance risk in
plant-disease datasets.

In [ ]:
y_true_onehot = label_binarize(clean["y_true"], classes=list(range(N_CLASSES)))

roc_auc_macro = roc_auc_score(y_true_onehot, clean["y_prob"], average="macro", multi_class="ovr")
roc_auc_weighted = roc_auc_score(y_true_onehot, clean["y_prob"], average="weighted", multi_class="ovr")
pr_auc_macro = average_precision_score(y_true_onehot, clean["y_prob"], average="macro")

print(f"[RESULT] Macro ROC-AUC (one-vs-rest):    {roc_auc_macro:.4f}")
print(f"[RESULT] Weighted ROC-AUC (one-vs-rest): {roc_auc_weighted:.4f}")
print(f"[RESULT] Macro PR-AUC (average precision): {pr_auc_macro:.4f}")

# Plot ROC curves for the 3 best- and 2 worst-performing classes by per-class AUC,
# rather than all 15 (illegible) or just one (uninformative).
per_class_auc = roc_auc_score(y_true_onehot, clean["y_prob"], average=None)
auc_ranked = sorted(zip(SELECTED_CLASSES, per_class_auc), key=lambda x: -x[1])
show_classes = auc_ranked[:3] + auc_ranked[-2:]

fig, ax = plt.subplots(figsize=(7, 6))
for cls_name, auc_val in show_classes:
    idx = SELECTED_CLASSES.index(cls_name)
    fpr, tpr, _ = roc_curve(y_true_onehot[:, idx], clean["y_prob"][:, idx])
    ax.plot(fpr, tpr, label=f"{cls_name}  (AUC={auc_val:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — 3 best-separated and 2 worst-separated classes")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

print("\n[RESULT] Per-class AUC — worst 3 (candidates for more training data / augmentation):")
for cls_name, auc_val in auc_ranked[-3:]:
    print(f"  {cls_name}: {auc_val:.4f}")


## 3.4 Top-k accuracy

In [ ]:
top1 = clean_accuracy
top2 = top_k_accuracy_score(clean["y_true"], clean["y_prob"], k=2, labels=list(range(N_CLASSES)))
top3 = top_k_accuracy_score(clean["y_true"], clean["y_prob"], k=3, labels=list(range(N_CLASSES)))

print(f"[RESULT] Top-1 accuracy: {top1:.4f}")
print(f"[RESULT] Top-2 accuracy: {top2:.4f}   (+{top2 - top1:.4f} over top-1)")
print(f"[RESULT] Top-3 accuracy: {top3:.4f}   (+{top3 - top1:.4f} over top-1)")
print("[INFO] A large top-1 -> top-3 gap indicates the correct class is usually a close")
print("[INFO] runner-up rather than completely missed -- relevant context for a")
print("[INFO] human-in-the-loop deployment (e.g. showing farmers the top-3 candidates).")


## 3.5 Concept-head evaluation
The 4-unit concept head exists in the architecture but `evaluate.py`'s original
scope only ever scored the disease head — this is the first evaluation of the
concept predictions themselves (multi-label: sigmoid + 0.5 threshold).

In [ ]:
concept_pred_binary = (clean["concept_prob"] >= 0.5).astype(int)
concept_true_binary = clean["concept_true"].astype(int)

subset_accuracy = float(np.mean(np.all(concept_pred_binary == concept_true_binary, axis=1)))
h_loss = hamming_loss(concept_true_binary, concept_pred_binary)

print(f"[RESULT] Concept subset accuracy (all 4 correct simultaneously): {subset_accuracy:.4f}")
print(f"[RESULT] Hamming loss (fraction of individual concept bits wrong): {h_loss:.4f}")
print()

concept_rows = []
for i, name in enumerate(CONCEPT_NAMES):
    p = precision_score(concept_true_binary[:, i], concept_pred_binary[:, i], zero_division=0)
    r = recall_score(concept_true_binary[:, i], concept_pred_binary[:, i], zero_division=0)
    f1 = f1_score(concept_true_binary[:, i], concept_pred_binary[:, i], zero_division=0)
    acc = float(np.mean(concept_true_binary[:, i] == concept_pred_binary[:, i]))
    concept_rows.append({"concept": name, "accuracy": acc, "precision": p, "recall": r, "f1": f1})

concept_df = pd.DataFrame(concept_rows)
print("[RESULT] Per-concept metrics:")
print(concept_df)

fig, ax = plt.subplots(figsize=(7, 4))
concept_df.set_index("concept")[["precision", "recall", "f1"]].plot.bar(ax=ax)
ax.set_title("Concept-head performance per concept")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 3.6 Corrupted test — robustness, precisely characterized
Beyond raw accuracy, Cohen's Kappa and Matthews Correlation Coefficient (MCC)
quantify agreement *beyond what random chance would produce* — the right
tools for asking "is this model doing anything at all under corruption, or
is it purely guessing?"

In [ ]:
import albumentations as A

transform_corrupt = A.Compose([
    A.MotionBlur(blur_limit=7, p=1.0),
    A.GaussNoise(p=1.0),
    A.RandomBrightnessContrast(brightness_limit=-0.3, contrast_limit=0.3, p=1.0),
])

def corrupt_fn(img):
    def _apply(img_np):
        return transform_corrupt(image=img_np.astype(np.uint8))["image"]
    out = tf.numpy_function(_apply, [tf.cast(img, tf.uint8)], tf.uint8)
    out.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return out

print("[OPERATION] Running full inference over the corrupted test set...")
ds_corrupt = get_tf_dataset(test_df, augment_fn=corrupt_fn)
corrupt = run_inference(model, ds_corrupt)

corrupt_accuracy = float(np.mean(corrupt["y_pred"] == corrupt["y_true"]))
kappa = cohen_kappa_score(corrupt["y_true"], corrupt["y_pred"])
mcc = matthews_corrcoef(corrupt["y_true"], corrupt["y_pred"])
chance_level = 1.0 / N_CLASSES

print(f"[RESULT] Corrupted test accuracy: {corrupt_accuracy:.4f}  (n={len(corrupt['y_true'])})")
print(f"[RESULT] Random-chance baseline:  {chance_level:.4f}")
print(f"[RESULT] Cohen's Kappa:  {kappa:.4f}   (0 = chance-level agreement, 1 = perfect)")
print(f"[RESULT] Matthews Corr.: {mcc:.4f}   (0 = chance-level, 1 = perfect, works well even with class imbalance)")

if kappa < 0.05:
    print("\n[FINDING] Kappa is essentially 0 -- the corrupted-image predictions carry no more")
    print("[FINDING] information than random guessing. This is a precise, defensible way to")
    print("[FINDING] state the robustness-track result: not just 'low accuracy' but")
    print("[FINDING] 'statistically indistinguishable from chance' under this corruption family.")
elif kappa < 0.20:
    print("\n[FINDING] Kappa indicates only slight agreement beyond chance.")
else:
    print("\n[FINDING] Kappa indicates the model retains meaningful signal under corruption.")


## 3.7 95% bootstrap confidence interval on clean test accuracy
With n=442 test images, a point-estimate accuracy alone doesn't convey
sampling uncertainty. 10,000 bootstrap resamples of the correct/incorrect
outcome vector give a defensible interval to quote.

In [ ]:
rng = np.random.default_rng(SEED)
correct = (clean["y_pred"] == clean["y_true"]).astype(int)
n = len(correct)
n_boot = 10_000

boot_accs = np.empty(n_boot)
for b in range(n_boot):
    idx = rng.integers(0, n, size=n)
    boot_accs[b] = correct[idx].mean()

ci_low, ci_high = np.percentile(boot_accs, [2.5, 97.5])
print(f"[RESULT] Clean test accuracy: {clean_accuracy:.4f}")
print(f"[RESULT] 95% bootstrap CI ({n_boot:,} resamples): [{ci_low:.4f}, {ci_high:.4f}]")
print(f"[RESULT] CI half-width: ±{(ci_high - ci_low) / 2:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(boot_accs, bins=50, color="#4C72B0", alpha=0.8)
ax.axvline(clean_accuracy, color="black", linestyle="-", label=f"point estimate {clean_accuracy:.3f}")
ax.axvline(ci_low, color="red", linestyle="--", label="95% CI")
ax.axvline(ci_high, color="red", linestyle="--")
ax.set_title(f"Bootstrap distribution of test accuracy (n={n})")
ax.set_xlabel("accuracy"); ax.legend()
plt.tight_layout()
plt.show()


## 3.8 Calibration — is the model's confidence trustworthy?
A reliability diagram bins predictions by their max softmax confidence and
plots empirical accuracy within each bin against the mean confidence in that
bin (a perfectly calibrated model sits on the diagonal). Brier score summarizes
the same idea in one number.

In [ ]:
confidences = clean["y_prob"].max(axis=1)
correct_mask = (clean["y_pred"] == clean["y_true"])

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_ids = np.digitize(confidences, bin_edges[1:-1])

bin_conf, bin_acc, bin_count = [], [], []
for b in range(n_bins):
    mask = bin_ids == b
    if mask.sum() == 0:
        continue
    bin_conf.append(confidences[mask].mean())
    bin_acc.append(correct_mask[mask].mean())
    bin_count.append(mask.sum())

bin_conf, bin_acc, bin_count = np.array(bin_conf), np.array(bin_acc), np.array(bin_count)
ece = float(np.sum(bin_count / n * np.abs(bin_acc - bin_conf)))

# Multiclass Brier score: mean squared error between predicted prob vector and one-hot truth.
brier = float(np.mean(np.sum((clean["y_prob"] - y_true_onehot) ** 2, axis=1)))

print(f"[RESULT] Expected Calibration Error (ECE, {n_bins} bins): {ece:.4f}")
print(f"[RESULT] Multiclass Brier score: {brier:.4f}  (lower is better, 0 = perfect)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax.plot(bin_conf, bin_acc, marker="o", color="#C44E52", label="model")
ax.set_xlabel("Mean predicted confidence"); ax.set_ylabel("Empirical accuracy")
ax.set_title("Reliability diagram — clean test set")
ax.legend()
plt.tight_layout()
plt.show()

if ece > 0.10:
    print("[FINDING] ECE > 0.10 -- the model is meaningfully over/under-confident;")
    print("[FINDING] raw softmax scores should not be presented to end users as calibrated")
    print("[FINDING] probabilities without temperature scaling or similar recalibration.")
else:
    print("[FINDING] Reasonably well calibrated -- softmax confidence is a fair proxy for")
    print("[FINDING] reliability at this operating point.")


## 3.9 INT8 post-training quantization + parity check

In [ ]:
tflite_path = "edge_model.tflite" if os.path.exists("manifest.csv") else "../edge_model.tflite"

if not os.path.exists(tflite_path):
    print("[OPERATION] Quantizing model to INT8...")
    train_df_q = manifest[manifest["split"] == "train"]
    calib_ds = get_tf_dataset(train_df_q, batch_size=32)

    def representative_data_gen():
        for imgs, _ in calib_ds.unbatch().batch(1).take(100):
            yield [imgs]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32
    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    print(f"[SUCCESS] Wrote {tflite_path}")
else:
    print(f"[LOADED] Existing quantized model at {tflite_path}")

fp32_mb = os.path.getsize(model_path) / 1e6
int8_mb = os.path.getsize(tflite_path) / 1e6
print(f"[RESULT] FP32 size: {fp32_mb:.2f} MB  ->  INT8 size: {int8_mb:.2f} MB  ({fp32_mb / int8_mb:.1f}x compression)")


In [ ]:
def evaluate_tflite(tflite_path, df):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()
    disease_output_idx = [d["index"] for d in output_details if d["shape"][-1] == N_CLASSES][0]

    y_true, y_pred = [], []
    for _, row in df.iterrows():
        img = _decode_resize(row["File_Path"]).numpy()[None, ...].astype(np.float32)
        interpreter.set_tensor(input_details["index"], img)
        interpreter.invoke()
        d_pred = interpreter.get_tensor(disease_output_idx)
        y_pred.append(int(np.argmax(d_pred[0])))
        y_true.append(int(row["label_idx"]))
    return np.array(y_true), np.array(y_pred)

print("[OPERATION] Running INT8 TFLite inference over the clean test set...")
tflite_y_true, tflite_y_pred = evaluate_tflite(tflite_path, test_df)
tflite_accuracy = float(np.mean(tflite_y_pred == tflite_y_true))

int8_delta = tflite_accuracy - clean_accuracy
print(f"[RESULT] INT8 TFLite accuracy: {tflite_accuracy:.4f}")
print(f"[RESULT] Delta vs FP32 clean: {int8_delta:+.4f}")

if abs(int8_delta) > 0.05:
    print("[FINDING] Quantization cost more than 5 accuracy points -- report this delta")
    print("[FINDING] alongside the compression ratio, not size alone.")
else:
    print("[FINDING] Quantization is near-lossless -- safe to present compression ratio alone.")


## 3.10 Inference latency benchmark — FP32 vs INT8
The actual "edge deployment" payoff isn't just file size — it's latency.
Benchmarks single-image inference (CPU) after a warm-up period, on both formats.

In [ ]:
N_WARMUP, N_RUNS = 5, 50
sample_imgs = [_decode_resize(p).numpy()[None, ...].astype(np.float32)
               for p in test_df["File_Path"].head(N_RUNS + N_WARMUP)]

# --- FP32 Keras latency ---
for img in sample_imgs[:N_WARMUP]:
    model.predict(img, verbose=0)
fp32_times = []
for img in sample_imgs[N_WARMUP:]:
    t0 = time.perf_counter()
    model.predict(img, verbose=0)
    fp32_times.append((time.perf_counter() - t0) * 1000)

# --- INT8 TFLite latency ---
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
for img in sample_imgs[:N_WARMUP]:
    interpreter.set_tensor(input_details["index"], img)
    interpreter.invoke()
int8_times = []
for img in sample_imgs[N_WARMUP:]:
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details["index"], img)
    interpreter.invoke()
    int8_times.append((time.perf_counter() - t0) * 1000)

fp32_mean, fp32_std = np.mean(fp32_times), np.std(fp32_times)
int8_mean, int8_std = np.mean(int8_times), np.std(int8_times)
speedup = fp32_mean / int8_mean

print(f"[RESULT] FP32 Keras : {fp32_mean:.2f} ± {fp32_std:.2f} ms/image  ({1000 / fp32_mean:.1f} img/s)")
print(f"[RESULT] INT8 TFLite: {int8_mean:.2f} ± {int8_std:.2f} ms/image  ({1000 / int8_mean:.1f} img/s)")
print(f"[RESULT] Speedup: {speedup:.2f}x  (CPU, single-image, n={N_RUNS} runs after {N_WARMUP} warmup)")
print("[NOTE] Absolute numbers are hardware-dependent -- the speedup ratio and compression")
print("[NOTE] ratio are the portable, resume-safe claims; do not quote raw ms without")
print("[NOTE] naming the CPU they were measured on.")


## 3.11 Summary — full metric sheet

In [ ]:
summary = pd.DataFrame([
    {"metric": "Clean test accuracy",                 "value": f"{clean_accuracy:.4f}"},
    {"metric": "95% bootstrap CI",                     "value": f"[{ci_low:.4f}, {ci_high:.4f}]"},
    {"metric": "Macro ROC-AUC (OvR)",                   "value": f"{roc_auc_macro:.4f}"},
    {"metric": "Macro PR-AUC",                          "value": f"{pr_auc_macro:.4f}"},
    {"metric": "Top-2 accuracy",                        "value": f"{top2:.4f}"},
    {"metric": "Top-3 accuracy",                        "value": f"{top3:.4f}"},
    {"metric": "Expected Calibration Error",            "value": f"{ece:.4f}"},
    {"metric": "Multiclass Brier score",                "value": f"{brier:.4f}"},
    {"metric": "Concept subset accuracy",               "value": f"{subset_accuracy:.4f}"},
    {"metric": "Concept Hamming loss",                  "value": f"{h_loss:.4f}"},
    {"metric": "Corrupted test accuracy",               "value": f"{corrupt_accuracy:.4f}"},
    {"metric": "Corrupted Cohen's Kappa",                "value": f"{kappa:.4f}"},
    {"metric": "Corrupted MCC",                          "value": f"{mcc:.4f}"},
    {"metric": "INT8 TFLite accuracy",                   "value": f"{tflite_accuracy:.4f}"},
    {"metric": "INT8 vs FP32 delta",                     "value": f"{int8_delta:+.4f}"},
    {"metric": "Compression ratio",                      "value": f"{fp32_mb / int8_mb:.1f}x"},
    {"metric": "INT8 vs FP32 latency speedup",            "value": f"{speedup:.2f}x"},
])
print(summary.to_string(index=False))

results_out_path = "eval_report/full_metrics.json" if os.path.exists("manifest.csv") else "../eval_report/full_metrics.json"
with open(results_out_path, "w") as f:
    json.dump(summary.set_index("metric")["value"].to_dict(), f, indent=2)
print(f"\n[SUCCESS] Full metric sheet saved to {results_out_path}")


## 3.12 Known limitations (feed these into the README verbatim)

- The 4 "concepts" are a deterministic re-encoding of the 15 class labels
  (every image of a class gets an identical concept vector) — an auxiliary
  hierarchical-label signal, not independently-annotated ground truth.
- Latency numbers in 3.10 are hardware-dependent; only the ratios are portable claims.
- See the printed **[FINDING]** lines above for the current honest interpretation
  of robustness and calibration results — copy them from this notebook's real
  output, not from memory.

**Next notebook:** `04_GradCAM_Explainability.ipynb`.
